# Упражнение: Цикл обучения как в PyTorch — на NumPy

## Задание

Напишите на NumPy полный цикл обучения нейросети — ровно то, что делает типичный код на PyTorch:

```python
model = nn.Sequential(nn.Linear(2, 3), nn.Tanh(), nn.Linear(3, 1))
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
loss_fn = nn.MSELoss()
for epoch in range(30):
    for xb, yb in batches:               # батчи 1, 2, 3 — без перемешивания
        optimizer.zero_grad()            # обнулить градиенты
        loss = loss_fn(model(xb), yb)    # forward + loss
        loss.backward()                  # backward: посчитать градиенты
        optimizer.step()                 # шаг SGD с моментом
```

Данные — `input.csv`: признаки `x1, x2`, цель `y` и номер батча `batch` (по 8 строк в батче).
Начальные веса — `weights.json` в раскладке PyTorch: `W1` (3 × 2), `b1` (3), `W2` (1 × 3), `b2` (1).

1. **Forward**: h = tanh(xb·W1ᵀ + b1), ŷ = h·W2ᵀ + b2.
2. **Loss**: MSE по батчу, L = mean((ŷ − y)²).
3. **Backward**: градиенты L по W1, b1, W2, b2 (цепное правило; tanh′ = 1 − tanh²).
4. **Step** (SGD с моментом μ = 0.9, lr = 0.1, как в PyTorch): для каждого параметра p с градиентом g
   v ← μ·v + g, p ← p − lr·v; в начале v = 0 (буфер момента **не** сбрасывается между батчами и эпохами).
5. Порядок в каждой эпохе: батч 1, батч 2, батч 3; строки внутри батча — в порядке файла.

**loss эпохи** — среднее трёх батчевых loss, посчитанных в forward этой эпохи (до соответствующих шагов) —
как `running_loss / len(loader)` в PyTorch.

## Подсказки

- Сначала напишите и проверьте forward и loss для первого батча при начальных весах, потом — backward.
- Проверьте градиенты численно: (L(p + ε) − L(p − ε)) / 2ε для пары элементов W1 — должно совпасть.
- Обратный проход: dŷ = 2(ŷ − y)/m, dW2 = dŷᵀ·h, db2 = Σdŷ, dh = dŷ·W2, dz = dh·(1 − h²),
  dW1 = dzᵀ·xb, db1 = Σdz. «zero_grad» в NumPy — это просто считать градиенты заново на каждом батче,
  а не прибавлять к старым.

## Ожидаемый формат выходного файла

CSV с колонками `epoch, loss` — 30 строк для эпох 1…30; `loss` округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/pytorch_full_guide
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.